# Export Model for Live Trading

Loads pre-computed features from `data/latest_features.jsonl`.
Uses optimal LR features from `data/optimal_features_lr.json` (generated by notebook 1).
Train/val split, train LogisticRegression, export to `models/`.

In [1]:
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

random.seed(42)
np.random.seed(42)

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)
FEATURES_PATH = Path("../data/latest_features.jsonl")
LR_FEATURES_PATH = Path("../data/optimal_features_lr.json")

## 1. Load pre-computed features

In [2]:
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

# Load optimal LR features
with open(LR_FEATURES_PATH) as _f:
    feat_cols = sorted(json.load(_f)["features"])
print(f"Using {len(feat_cols)} optimal LR features from {LR_FEATURES_PATH.name}")

df[feat_cols] = df[feat_cols].fillna(0.0)

print(f"Loaded {len(df):,} rows, {df['candle_id'].nunique()} candles")
print(f"UP rate: {df['target'].mean() * 100:.1f}%")

Loaded 53,377 rows, 1103 candles
Using 31 optimal features (from notebook 3 Logistic selection)
UP rate: 51.8%


## 2. Train/validation split (80/20 by time)

In [3]:
candle_ids = df["candle_id"].unique()
split_idx = int(len(candle_ids) * 0.8)
train_ids = set(candle_ids[:split_idx])
val_ids = set(candle_ids[split_idx:])

df_train = df[df["candle_id"].isin(train_ids)]
df_val = df[df["candle_id"].isin(val_ids)]

print(f"Train: {len(df_train):,} rows, {df_train['candle_id'].nunique()} candles")
print(f"Val:   {len(df_val):,} rows, {df_val['candle_id'].nunique()} candles")

Train: 42,925 rows, 882 candles
Val:   10,452 rows, 221 candles


## 3. Train & evaluate

In [4]:
scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[feat_cols].values)
y_train = df_train["target"].values

X_val = scaler.transform(df_val[feat_cols].values)
y_val = df_val["target"].values

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
val_acc = accuracy_score(y_val, model.predict(X_val))

print(f"Train accuracy: {train_acc * 100:.1f}%")
print(f"Val accuracy:   {val_acc * 100:.1f}%")
print("\nValidation report:")
print(classification_report(y_val, model.predict(X_val), target_names=["DOWN", "UP"]))

Train accuracy: 75.9%
Val accuracy:   78.3%

Validation report:
              precision    recall  f1-score   support

        DOWN       0.80      0.79      0.79      5541
          UP       0.77      0.77      0.77      4911

    accuracy                           0.78     10452
   macro avg       0.78      0.78      0.78     10452
weighted avg       0.78      0.78      0.78     10452



## 4. Export (retrained on all data)

In [5]:
X_all = scaler.fit_transform(df[feat_cols].values)
y_all = df["target"].values
model.fit(X_all, y_all)

joblib.dump(model, MODELS_DIR / "logistic_v1.joblib")
joblib.dump(scaler, MODELS_DIR / "scaler_v1.joblib")
joblib.dump(feat_cols, MODELS_DIR / "feature_cols_v1.joblib")

print(f"Saved to {MODELS_DIR} (retrained on all {len(df):,} rows):")
for f in sorted(MODELS_DIR.glob("*_v1.*")):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

Saved to ../models (retrained on all 53,377 rows):
  feature_cols_v1.joblib (619 bytes)
  logistic_v1.joblib (1,119 bytes)
  rf_feature_cols_v1.joblib (373 bytes)
  rf_scaler_v1.joblib (1,063 bytes)
  rf_v1.joblib (21,996,649 bytes)
  scaler_v1.joblib (1,311 bytes)


## 5. Verify load

In [6]:
m2 = joblib.load(MODELS_DIR / "logistic_v1.joblib")
s2 = joblib.load(MODELS_DIR / "scaler_v1.joblib")
fc2 = joblib.load(MODELS_DIR / "feature_cols_v1.joblib")

sample = df[fc2].iloc[:1].fillna(0.0).values
prob_orig = model.predict_proba(scaler.transform(sample))[0, 1]
prob_loaded = m2.predict_proba(s2.transform(sample))[0, 1]

print(f"Original prob:  {prob_orig:.6f}")
print(f"Loaded prob:    {prob_loaded:.6f}")
print(f"Match: {prob_orig == prob_loaded}")
print(f"\nFeature columns ({len(fc2)}): {fc2[:5]} ... {fc2[-5:]}")

Original prob:  0.462434
Loaded prob:    0.462434
Match: True

Feature columns (31): ['bollinger_pct_b', 'btc_direction_consistency', 'btc_move_from_open', 'btc_token_correlation', 'btc_velocity'] ... ['up_book_imbalance', 'up_spread_level', 'volume_momentum', 'volume_price_correlation', 'weighted_mid_price']
